<a href="https://colab.research.google.com/github/rashidiqra710-web/base-model-before-alignment/blob/main/before_Alignment_of_Base_model_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U trl transformers datasets accelerate matplotlib pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [2]:
!pip install "pandas==2.2.3" "pyarrow<24"

  Using cached pandas-2.2.3-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (89 kB)
  Using cached pyarrow-23.0.1-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (3.1 kB)
Using cached pandas-2.2.3-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.7 MB)
Using cached pyarrow-23.0.1-cp313-cp313-manylinux_2_28_x86_64.whl (47.6 MB)
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 26.0.1
    Uninstalling pyarrow-26.0.1:
      Successfully uninstalled pyarrow-26.0.1
  Attempting uninstall: pandas
    Found existing installation: pandas 3.1.1
    Uninstalling pandas-3.1.1:
      Successfully uninstalled pandas-3.1.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 5.2.0 requires pyarrow>=24.0.0, but you have pyarrow 23.0.1 which is incompatible.


# Experiment Design

## Question
Does DPO make the Qwen2.5-0.5B model give better answers?

## Prompts (6 prompts, 4 types)
1. Explanation: Explain what a DPO means?
2. Explanation: What is the difference between software and hardware?
3. Advice: I have an assignment to complete within a week on RLHF so what should i do and which model should i select an which dataset should i select according to the topic ?
4. Professional writing: Write a breif description on University of Gwadar.
5. Reasoning: “A shop sells 3 notebooks for 150 rupees. How much will 5 notebooks cost? Explain step by step.” Needs two steps: price of one notebook (50), then 5 notebooks (250 rupees).
6. Question answering: What is the capital of Balochistan and why was it chosen as capital city among Balochistan?

## How I will score
I will give each answer 1 to 5 marks for:
- Helpfulness
- Relevance
- Clarity
- Instruction following
Total = 20 marks per answer.

## Why this method
It is simple. I use the same rules for every answer, so the comparison is fair.

## Beta values
- Run A: beta = 0.1 (required)
- Run B: beta = 0.5 (to see if the model stays closer to the base model)

## Preference data
I will look at 5 examples from the dataset.

In [3]:
import torch, pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "Qwen/Qwen2.5-0.5B"
tok = AutoTokenizer.from_pretrained(name)
base = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16).to("cuda")

prompts = [
 "Explain what DPO means.",
 "What is the difference between software and hardware?",
 "I have an assignment to complete within a week on RLHF. What should I do, which model should I select and which dataset should I select for this topic?",
 "Write a brief description of the University of Gwadar.",
 "A shop sells 3 notebooks for 150 rupees. How much will 5 notebooks cost? Explain step by step.",
 "What is the capital of Balochistan and why was it chosen as the capital city?",
]

def generate(model, prompt):
    text = tok.apply_chat_template([{"role":"user","content":prompt}],
                                   tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors="pt").to("cuda")
    out = model.generate(**inp, max_new_tokens=200, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)

base_out = {p: generate(base, p) for p in prompts}
for p, r in base_out.items(): print("PROMPT:", p, "\nBASE:", r, "\n", "-"*60)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

PROMPT: Explain what DPO means. 
BASE: DPO stands for Data Protection Officer. It is a person who is responsible for ensuring that personal data is protected and handled in a way that complies with data protection laws and regulations. The DPO is responsible for ensuring that the company's data protection policies are being followed, and that any data breaches or other issues are addressed promptly and effectively. The DPO is also responsible for ensuring that the company's data is stored securely and that any data that is collected is only used for the intended purpose. 
 ------------------------------------------------------------
PROMPT: What is the difference between software and hardware? 
BASE: oftware refers to the software that is installed on a computer, while hardware refers to the physical components of a computer, such as the CPU, RAM, and hard drive.yük
ѐ
oftware
oftware
oftware
oftware
oftware
oftware
oftware
oftware
oftware
oftware
oftware
oftware
oftware
oftware
oftware

In [5]:
import sys
import torch
from transformers import TextIteratorStreamer
from threading import Thread

def generate_interactive(model, tokenizer):
    user_prompt = input("Enter your prompt: ")

    # Apply chat template
    formatted_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_prompt}],
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize
    inputs = tokenizer([formatted_text], return_tensors="pt").to("cuda")

    # Setup a streamer for real-time token-by-token output
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    generation_kwargs = dict(
        inputs,
        streamer=streamer,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    # Run generation in a separate thread so we can stream tokens in the main thread
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    print("\n--- Generating Output Token-by-Token ---")
    for new_text in streamer:
        print(new_text, end="", flush=True)
    print("\n----------------------------------------")

# Run the interactive generation function
generate_interactive(base, tok)

Enter your prompt: what is english?

--- Generating Output Token-by-Token ---
English is a language that is spoken in many countries around the world. It is a written and spoken language that uses the Latin alphabet. English is one of the most widely spoken languages in the world, and it is used in many different fields, including business, education, and communication. English is also used in many different forms of media, such as television, radio, and film. English is a language that is important for understanding and communicating with people from different countries and cultures.
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey
 Comey

----------------------------------------


In [4]:
from datasets import load_dataset
ds = load_dataset("trl-lib/ultrafeedback_binarized", split="train")
print(ds.column_names, len(ds))

for i in range(5):
    ex = ds[i]
    print("PROMPT  :", ex["chosen"][0]["content"][:300])
    print("CHOSEN  :", ex["chosen"][-1]["content"][:400])
    print("REJECTED:", ex["rejected"][-1]["content"][:400])
    print("SCORES  :", ex["score_chosen"], "vs", ex["score_rejected"], "\n", "="*60)

README.md:   0%|          | 0.00/643 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  131MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.14MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/62135 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

['chosen', 'rejected', 'score_chosen', 'score_rejected'] 62135
PROMPT  : Use the pygame library to write a version of the classic game Snake, with a unique twist
CHOSEN  : Sure, I'd be happy to help you write a version of the classic game Snake using the pygame library! Here's a basic outline of how we can approach this:

1. First, we'll need to set up the game display and create a game object that we can use to handle the game's state.
2. Next, we'll create the game's grid, which will be used to represent the game board. We'll need to define the size of the grid an
REJECTED: Sure, here's an example of how to write a version of Snake game with a unique twist using the Pygame library:
```python
import pygame

class SnakeGame:
    def __init__(self, game_width, game_height):
        pygame.init()
        screen = pygame.display.set_mode((game_width, game_height))
        pygame.display.set_caption("Snake Game")
        self.speed = 5  # Speed of the snake
        self.fo
SCORES  : 6.0 vs